In [ ]:
#----------------------------------------
# Step 1 — install sb3
#----------------------------------------

# Run this once if packages are missing
#%pip install "stable-baselines3[extra]" gymnasium torch --quiet

#%pip install "numpy<2" --force-reinstall

In [1]:
import numpy as np
import stable_baselines3
import matplotlib

print("NumPy:", np.__version__)
print("Stable-Baselines3 imported successfully")

NumPy: 1.26.4
Stable-Baselines3 imported successfully


In [3]:
#----------------------------------------
# Step 2 — Use this corrected DDPG wrapper
#----------------------------------------


%run Maze_1_ML.ipynb
import gymnasium as gym
from gymnasium import spaces
import numpy as np


class ContinuousMazeWrapper(gym.Env):
    def __init__(self):
        super().__init__()

        self.env = MazeEnv()

        # Same observation space as your original MazeEnv
        self.observation_space = self.env.observation_space

        # DDPG needs a continuous action space
        # We use one continuous value between -1 and 1
        self.action_space = spaces.Box(
            low=np.array([-1.0], dtype=np.float32),
            high=np.array([1.0], dtype=np.float32),
            dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        obs, info = self.env.reset(seed=seed, options=options)
        return obs, info

    def step(self, action):
        # Convert continuous DDPG action into one of 4 maze actions
        a = float(np.asarray(action).reshape(-1)[0])

        if a < -0.5:
            discrete_action = 0   # up
        elif a < 0.0:
            discrete_action = 1   # down
        elif a < 0.5:
            discrete_action = 2   # left
        else:
            discrete_action = 3   # right

        obs, reward, terminated, truncated, info = self.env.step(discrete_action)

        info["continuous_action"] = a
        info["discrete_action"] = discrete_action

        return obs, reward, terminated, truncated, info

    def render(self):
        self.env.render()

    def close(self):
        pass
        
        
        
print("Done!")


Done!


In [4]:
#----------------------------------------
# Step 3 - Test the environment
#----------------------------------------

env = ContinuousMazeWrapper()

obs, info = env.reset()
print("Initial observation:", obs)

for i in range(10):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)

    print(
        f"Step {i+1}: "
        f"continuous={info['continuous_action']:.2f}, "
        f"discrete={info['discrete_action']}, "
        f"obs={obs}, reward={reward}, done={terminated or truncated}"
    )

    if terminated or truncated:
        obs, info = env.reset()

Initial observation: [0. 0.]
Step 1: continuous=0.39, discrete=2, obs=[0. 0.], reward=-0.1, done=False
Step 2: continuous=0.39, discrete=2, obs=[0. 0.], reward=-0.1, done=False
Step 3: continuous=0.17, discrete=2, obs=[0. 0.], reward=-0.1, done=False
Step 4: continuous=0.32, discrete=2, obs=[0. 0.], reward=-0.1, done=False
Step 5: continuous=0.10, discrete=2, obs=[0. 0.], reward=-0.1, done=False
Step 6: continuous=0.58, discrete=3, obs=[0.         0.06666667], reward=-0.1, done=False
Step 7: continuous=0.45, discrete=2, obs=[0. 0.], reward=-0.1, done=False
Step 8: continuous=0.25, discrete=2, obs=[0. 0.], reward=-0.1, done=False
Step 9: continuous=1.00, discrete=3, obs=[0.         0.06666667], reward=-0.1, done=False
Step 10: continuous=-0.62, discrete=0, obs=[0.         0.06666667], reward=-0.1, done=False


In [5]:
#----------------------------------------
# Step 4 — Train DDPG
#----------------------------------------

from stable_baselines3 import DDPG
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor
import os

env = ContinuousMazeWrapper()

check_env(env, warn=True)

env = Monitor(env)

model = DDPG(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=1e-3,
    buffer_size=50_000,
    batch_size=64,
    gamma=0.99,
)

model.learn(total_timesteps=20_000)

os.makedirs("models", exist_ok=True)
model.save("models/ddpg_maze_quick")

print("Model saved.")

Using cpu device
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 500      |
|    ep_rew_mean     | -50      |
| time/              |          |
|    episodes        | 4        |
|    fps             | 50       |
|    time_elapsed    | 39       |
|    total_timesteps | 2000     |
| train/             |          |
|    actor_loss      | 0.98     |
|    critic_loss     | 9.24e-05 |
|    learning_rate   | 0.001    |
|    n_updates       | 1899     |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 500      |
|    ep_rew_mean     | -50      |
| time/              |          |
|    episodes        | 8        |
|    fps             | 47       |
|    time_elapsed    | 83       |
|    total_timesteps | 4000     |
| train/             |          |
|    actor_loss      | 1.78     |
|    critic_loss     | 0.000273 |
|    learning_rate   | 0.001  

In [6]:
#----------------------------------------
# Step 5 - Test the trained model
#----------------------------------------

from stable_baselines3 import DDPG

env = ContinuousMazeWrapper()

model = DDPG.load("models/ddpg_maze_quick", env=env)

def run_episode(model, env, deterministic=True, max_steps=500, render=True):
    obs, info = env.reset()
    total_reward = 0

    for step in range(max_steps):
        action, _ = model.predict(obs, deterministic=deterministic)

        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward

        if render:
            env.render()

        if terminated or truncated:
            break

    return total_reward, step + 1, terminated, truncated


for episode in range(3):
    total_reward, steps, terminated, truncated = run_episode(
        model,
        env,
        deterministic=True,
        render=True
    )

    print(
        f"Episode {episode+1}: "
        f"reward={total_reward:.2f}, "
        f"steps={steps}, "
        f"terminated={terminated}, "
        f"truncated={truncated}"
    )

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Player at: (1, 0)
Player at: (2, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Player at: (3, 0)
Playe

In [7]:
from stable_baselines3 import DDPG
import time
from IPython.display import clear_output

# Create environment
env = ContinuousMazeWrapper()

# Load trained model
model = DDPG.load("models/ddpg_maze_quick", env=env)

# Reset environment
obs, info = env.reset()

# Watch one episode
for step in range(500):
    action, _ = model.predict(obs, deterministic=True)

    obs, reward, terminated, truncated, info = env.step(action)

    clear_output(wait=True)
    env.render()

    print("Step:", step + 1)
    print("Reward:", reward)
    print("Done:", terminated or truncated)

    time.sleep(0.1)

    if terminated or truncated:
        break

Player at: (3, 0)
Step: 500
Reward: -0.1
Done: True


In [8]:
import os
os.getcwd()

'C:\\Users\\nagyo\\Machine Learning\\Maze'